# Mental Health BERT Fine-Tuned – Kaggle Evaluation Notebook

In [ ]:
pip install transformers datasets scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

In [ ]:
train_df = pd.read_csv("/content/mental_heath_unbanlanced.csv")
test_df = pd.read_csv("/content/mental_health_combined_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

In [ ]:
train_df = train_df[['text','status']]
test_df = test_df[['text','status']]

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

print(train_df.head())

In [ ]:
label_encoder = LabelEncoder()

train_df['label'] = label_encoder.fit_transform(train_df['status'])
test_df['label'] = label_encoder.transform(test_df['status'])

num_labels = len(train_df['label'].unique())

print("Number of classes:", num_labels)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=0.1,
    random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
class MentalHealthDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = MentalHealthDataset(train_texts, train_labels, tokenizer)
val_dataset = MentalHealthDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * 3

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [ ]:
from tqdm import tqdm

epochs = 5

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Loss:", avg_loss)

In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

In [ ]:
print(classification_report(true_labels, predictions))

In [ ]:
model.save_pretrained("mental_health_bert_model")
tokenizer.save_pretrained("mental_health_bert_model")

In [ ]:
!pip install gradio transformers torch --quiet

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "mental_health_bert_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

labels = ["Anxiety", "Depression", "Normal", "Suicidal"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
import re

def predict(text):

    if text is None or text.strip() == "":
        return "Please enter text", {}

    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower().strip()

    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    probs = F.softmax(outputs.logits, dim=1)[0].cpu()

    pred = torch.argmax(probs).item()
    confidence = probs[pred].item() * 100

    result = {labels[i]: float(probs[i]) for i in range(len(labels))}

    message = f"Prediction: {labels[pred]} | Confidence: {confidence:.2f}%"

    return message, result

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🧠 AI Mental Health Detection System
    ### BERT-based NLP Model

    Enter a sentence to analyze emotional mental health condition.
    """)

    with gr.Row():

        with gr.Column(scale=1):

            text_input = gr.Textbox(
                label="Enter Text",
                placeholder="Example: I feel very stressed and anxious today...",
                lines=4
            )

            analyze_btn = gr.Button("Analyze Mental Health")

        with gr.Column(scale=1):

            prediction_output = gr.Textbox(label="Prediction Result")

            probability_output = gr.Label(label="Class Probabilities")

    analyze_btn.click(
        fn=predict,
        inputs=text_input,
        outputs=[prediction_output, probability_output]
    )

    gr.Markdown("""
    ### Mental Health Categories

    🟡 **Anxiety** – Stress, fear, excessive worry
    🔵 **Depression** – Sadness, hopelessness
    🟢 **Normal** – Healthy emotional state
    🔴 **Suicidal** – Self-harm related thoughts
    """)

demo.launch(share=True)

In [ ]:
!zip -r mental_health_bert_model.zip mental_health_bert_model